In [0]:
import pandas as pd

# Load pre-cleaned data
df_memb = pd.read_parquet("/Volumes/workspace/default/retail_data/df_memb.parquet")
df_guest = pd.read_parquet("/Volumes/workspace/default/retail_data/df_guest.parquet")

print(f"Data loaded — Members: {len(df_memb):,} | Guests: {len(df_guest):,}")

In [0]:
# Install visualisation libraries
%pip install matplotlib seaborn plotly

# Import all libraries we need for EDA and visualisation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime

# Set visual style for all matplotlib charts
sns.set_theme(style="darkgrid")

# Make charts display inline in the notebook
%matplotlib inline

print("Libraries imported successfully")

In [0]:
# KPI SUMMARY
# First thing any business wants to know — the big numbers at a glance
# ------------------------------------------

# Calculate KPIs
total_revenue = df_memb['TotalRevenue'].sum() + df_guest['TotalRevenue'].sum()
member_revenue = df_memb['TotalRevenue'].sum()
guest_revenue = df_guest['TotalRevenue'].sum()
total_orders = df_memb['Invoice'].nunique() + df_guest['Invoice'].nunique()
total_customers = df_memb['Customer ID'].nunique()
total_products = df_memb['Description'].nunique()
avg_order_value = df_memb['TotalRevenue'].groupby(df_memb['Invoice']).sum().mean()
avg_revenue_per_customer = member_revenue / total_customers

print("BUSINESS KPI SUMMARY")
print("-" * 45)
print(f"Total Revenue: £{total_revenue:>12,.2f}")
print(f"Member Revenue: £{member_revenue:>12,.2f}")
print(f"Guest Revenue: £{guest_revenue:>12,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Unique Members: {total_customers:,}")
print(f"Unique Products: {total_products:,}")
print(f"Avg Order Value: £{avg_order_value:,.2f}")
print(f"Avg Revenue per Customer: £{avg_revenue_per_customer:,.2f}")

In [0]:
# REVENUE TREND OVER TIME
# Shows how revenue grew or declined
# month by month over the 2 year period
# --------------------------------------

# Extract year and month from InvoiceDate
df_memb['Year'] = df_memb['InvoiceDate'].dt.year
df_memb['Month'] = df_memb['InvoiceDate'].dt.month
df_memb['YearMonth'] = df_memb['InvoiceDate'].dt.to_period('M').astype(str)

# Group revenue by month
monthly_revenue = df_memb.groupby('YearMonth')['TotalRevenue'].sum().reset_index()
monthly_revenue.columns = ['YearMonth', 'Revenue']

# Sort chronologically
monthly_revenue = monthly_revenue.sort_values('YearMonth')

In [0]:
# Plot
plt.figure(figsize=(14, 6))
plt.plot(
    monthly_revenue['YearMonth'],
    monthly_revenue['Revenue'],
    color='#4C72B0',
    linewidth=2.5,
    marker='o',
    markersize=5
)

# Highlight the peak month
peak_month = monthly_revenue.loc[monthly_revenue['Revenue'].idxmax()]
plt.annotate(
    f"Peak: £{peak_month['Revenue']:,.0f}",
    xy=(peak_month['YearMonth'], peak_month['Revenue']),
    xytext=(10, 10),
    textcoords='offset points',
    fontsize=10,
    color='red'
)

plt.title('Monthly Revenue Trend (2009-2011)', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Revenue (£)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Print monthly numbers too
print("\nMONTHLY REVENUE BREAKDOWN")
print(monthly_revenue.to_string(index=False))

In [0]:
# TOP COUNTRIES BY REVENUE
# Shows which markets drive the most revenue
# -------------------------------------------

# Group revenue by country
country_revenue = df_memb.groupby('Country')['TotalRevenue'].sum().reset_index()
country_revenue.columns = ['Country', 'Revenue']

# Sort by revenue descending and take top 10
country_revenue = country_revenue.sort_values('Revenue', ascending=False).head(10)

# Calculate percentage of total member revenue
country_revenue['Percentage'] = (
    country_revenue['Revenue'] / df_memb['TotalRevenue'].sum() * 100
).round(2)


In [0]:
# Plot
plt.figure(figsize=(12, 6))
bars = plt.barh(
    country_revenue['Country'],
    country_revenue['Revenue'],
    color='#4C72B0'
)

# Add value labels on each bar
for bar, pct in zip(bars, country_revenue['Percentage']):
    plt.text(
        bar.get_width() + 50000,
        bar.get_y() + bar.get_height() / 2,
        f'£{bar.get_width():,.0f} ({pct}%)',
        va='center',
        fontsize=9
    )

plt.title('Top 10 Countries by Revenue', fontsize=16, fontweight='bold')
plt.xlabel('Revenue (£)', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Print the numbers too
print("\nTOP 10 COUNTRIES BY REVENUE")
print(country_revenue.to_string(index=False))